In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../")

from ease_recommender import *
from npmi_recommender import *

import pickle as p

def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

loading cache data...
building csr matrices...
done


In [2]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

f_name = find_match_using_terms(["Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)"], cat2idx)
f = cat2idx[f_name]
print(f"{f_name=}")
print(f"Num Rows: {artist_mat[:, f].sum()}")

a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412
c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71
d_name='Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb)'
Num Rows: 12615
e_name='Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)'
Num Rows: 32356


In [4]:
mat = artist_mat
mat = csr_array(mat)

n_users, n_items = mat.shape

X = mat.T @ mat
# X = X / n_users

X.shape

(295860, 295860)

In [5]:
X2 = X / n_users

In [6]:
counts = X.diagonal()

In [15]:
relevant = counts >= counts[[a,b,c,d,e,f]].min()
perc_relevant = relevant.mean()

X.shape[0] * perc_relevant

np.float64(25950.999999999996)

In [16]:
X[relevant].shape

(25951, 295860)

In [17]:
%%time

from sklearn.feature_extraction.text import TfidfTransformer
from scipy.sparse import csr_matrix

# tfidf = TfidfTransformer(
# #     norm='l1',
# #     use_idf=False,
# #     smooth_idf=False,
#     sublinear_tf=True)

# Y = tfidf.fit_transform(X)

CPU times: total: 15.6 ms
Wall time: 13.1 ms


In [22]:
# assert np.allclose(Y[a].toarray(), tfidf.transform(X[a:a+1]).toarray()[0])

In [23]:
# def get_metric(i, j, Y):
#     return np.argsort(-Y[i].toarray()).tolist().index(j)

# metrics = [
# #     get_metric(a, b, Y),
# #     get_metric(b, a, Y),
    
#     get_metric(b, c, Y),
#     get_metric(c, b, Y),
# ]

# np.mean(metrics)

In [24]:
for idx in np.argsort(-Y[e].toarray())[:20]:
    print(idx2cat[idx])

S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)
Justin Vernon (spotify:artist:13rHmjtJmlIJ2aDyJc7CLV)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Isbells (spotify:artist:14dULnNGmLKnS59BzNrHi4)
The Middle East (spotify:artist:6imbHAlhHrFwtsOgqpeBK2)
Whitley (spotify:artist:17C909ZAaDFHCNLhsqsb25)
Winter Aid (spotify:artist:15S89CUJtshT2P7WIa2M5l)
J. Tillman (spotify:artist:21XbnrbEMUTZelIfoV12hC)
Bon Iver & St. Vincent (spotify:artist:5NexFJlYHMb5IC6z6ci3BA)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)
Gem Club (spotify:artist:7mfGqyAztYr0FI5gK5OCNp)
Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)
Horse Feathers (spotify:artist:0lO2c86rQmrRJArBxgw0v8)
Axel Flovent (spotify:artist:6jn7W8NuX94FWZyeGlyCaJ)
The Milk Carton Kids (spotify:artist:7fxtWEwKKrFaykKItspdYg)
The Black Atlantic (spotify:artist:33kge1mmCkHoYWJ4kJe6BC)
Ciaran Lavery (spotify:artist:7zOuMHqRJ6YOMnCGpLfuTU)
Dry the River (spotify:artist:5VIq5RHAb

In [ ]:
# TODO: add correlation to compare rankings etc.

In [18]:
from sklearn.feature_extraction.text import TfidfTransformer
from scipy.sparse import csr_matrix

def get_metric(i, j, Y):
    return np.argsort(-Y[i].toarray()).tolist().index(j)

In [19]:
tfidf = TfidfTransformer(sublinear_tf=True)

In [35]:
tfidf.fit(mat)
Y = tfidf.transform(X)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

35.2
33.1
32.8


np.float64(7.0)

In [36]:
Y = tfidf.fit_transform(X)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

37.0
33.2
32.9


np.float64(9.0)

In [ ]:
Y = X2

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.tocoo().nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [ ]:
alpha = .1 * .85

Y = sparse_laplace_sppmi(X, alpha=alpha)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
alpha = .1 * .85

Y = sparse_laplace_sppmi(X, alpha=alpha, normalize=True, zero_diag=False)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
alpha = 0

Y = sparse_laplace_sppmi(X, alpha=alpha, zero_diag=False)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
lambda_ = 100

Yb = calculate_ease_for_item_cg(mat, b, lambda_)
Yc = calculate_ease_for_item_cg(mat, c, lambda_)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Yb,
        "b": Yc,
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
    np.argsort(-Yb).tolist().index(c),
    np.argsort(-Yc).tolist().index(b)
]

np.mean(metrics)

In [ ]:
lambda_ = 1000

Yb = calculate_ease_for_item_cg(mat, b, lambda_)
Yc = calculate_ease_for_item_cg(mat, c, lambda_)

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Yb,
        "b": Yc,
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
    np.argsort(-Yb).tolist().index(c),
    np.argsort(-Yc).tolist().index(b)
]

np.mean(metrics)

In [21]:
import numpy as np
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, check_array

class BM25Transformer(BaseEstimator, TransformerMixin):
    """
    BM25 transformer compatible with scikit-learn.
    
    Implements the Okapi BM25 ranking function using the Lucene/Elasticsearch 
    standard IDF formula to ensure non-negative weights.
    
    Parameters
    ----------
    k1 : float, default=1.5
        Term frequency saturation parameter. Higher values make the score 
        grow faster with term frequency.
        
    b : float, default=0.75
        Length normalization parameter. 1.0 is full normalization, 
        0.0 is no normalization.
    """
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, X, y=None):
        """
        Learn the idf vector and average document length.
        """
        # Accept sparse matrix (CSR/CSC) or dense
        X = check_array(X, accept_sparse=['csr', 'csc'], dtype=np.float64)
        
        n_samples, n_features = X.shape
        
        # 1. Calculate Average Document Length
        # Sum along axis 1 (columns) to get total words per doc
        doc_lens = np.array(X.sum(axis=1)).flatten()
        self.avg_dl_ = np.mean(doc_lens)
        
        # 2. Calculate IDF (Lucene/Standard formulation)
        # DF: Number of documents containing the term
        doc_freqs = np.array((X > 0).sum(axis=0)).flatten()
        
        # Formula: log(1 + (N - n + 0.5) / (n + 0.5))
        # The "+1" inside the log ensures IDF is always positive.
        self.idf_ = np.log(1 + (n_samples - doc_freqs + 0.5) / (doc_freqs + 0.5))
        
        # For strict sklearn compatibility, we should store n_features_in_
        self.n_features_in_ = n_features
        return self

    def transform(self, X):
        """
        Transform count matrix to BM25 weighted matrix.
        """
        check_is_fitted(self, ['idf_', 'avg_dl_'])
        # Copy=True is crucial to avoid modifying the input matrix in place
        X = check_array(X, accept_sparse='csr', copy=True, dtype=np.float64)
        
        n_samples, n_features = X.shape
        if n_features != self.n_features_in_:
            raise ValueError(f"X has {n_features} features, but BM25Transformer "
                             f"is expecting {self.n_features_in_} features.")

        # 1. Calculate document lengths for the input matrix
        doc_lens = np.array(X.sum(axis=1)).flatten()

        # 2. Calculate the length normalization factor per document
        # Factor = (1 - b + b * |D| / avg_dl)
        len_norm = (1 - self.b) + self.b * (doc_lens / self.avg_dl_)
        
        # 3. Apply TF saturation with length normalization
        # The sparse operation is:
        # Score = IDF * (TF * (k1 + 1)) / (TF + k1 * len_norm)
        
        # We need to map the document-level len_norm to every non-zero entry in X.
        # X.indptr tells us where each row starts and ends in X.data
        row_repeats = np.diff(X.indptr)
        expanded_len_norm = np.repeat(len_norm, row_repeats)
        
        # Apply the BM25 TF formula directly on the sparse data array
        # This is highly efficient vectorized operation
        numerator = X.data * (self.k1 + 1)
        denominator = X.data + (self.k1 * expanded_len_norm)
        X.data = numerator / denominator
        
        # 4. Multiply by IDF
        # Element-wise multiplication of columns by diagonal IDF matrix
        X = X @ sp.diags(self.idf_)
        
        return X

In [22]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import CountVectorizer

print("--- 1. Setup Data ---")
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the dog barked at the mailman",
    "foxes are quick and brown",
    "the lazy dog sleeps all day",
    "rabbits and foxes live in the forest"
]

# Use whitespace tokenization for total control
tokenizer = lambda x: x.split()
vectorizer = CountVectorizer(tokenizer=tokenizer, token_pattern=None)
X_counts = vectorizer.fit_transform(corpus)

print("--- 2. Run Our BM25Transformer ---")
# Initialize with standard params
bm25_trans = BM25Transformer(k1=1.5, b=0.75)
X_bm25 = bm25_trans.fit_transform(X_counts)

print("--- 3. Run rank_bm25 (Ground Truth) ---")
tokenized_corpus = [doc.split() for doc in corpus]
bm25_lib = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

# CRITICAL: Align IDFs for valid comparison.
# rank_bm25 uses a formula that allows negative weights.
# We overwrite its IDF dictionary with our non-negative values 
# to verify the MATRIX LOGIC (TF saturation + Length Norm).
vocab_map = vectorizer.vocabulary_
for word, idx in vocab_map.items():
    if word in bm25_lib.idf:
        bm25_lib.idf[word] = bm25_trans.idf_[idx]

print("--- 4. Verify Specific Terms ---")

# Let's check the term "dog"
query = "dog"
term_idx = vocab_map[query]

# Get scores from Library (calculated query-by-query)
# rank_bm25 sums scores for all words in query. For single word, it's just that word's weight.
lib_scores = bm25_lib.get_scores([query])

# Get scores from Our Matrix (column slice)
our_scores = X_bm25.getcol(term_idx).toarray().flatten()

print(f"Comparison for term '{query}':")
print(f"Library: {lib_scores}")
print(f"Ours:    {our_scores}")

np.testing.assert_allclose(
    our_scores, lib_scores, rtol=1e-6,
    err_msg="Scores for single term do not match!"
)
print("✅ Single term match successful.")

print("\n--- 5. Verify Full Matrix Consistency ---")
# We reconstruct the document scores for EVERY term and compare

# Iterate over every document and every term to check accuracy
X_dense = X_bm25.toarray()

# We will compute the expected weight for every non-zero element manually
# using the objects from rank_bm25 (which now shares our IDF)
rows, cols = X_counts.nonzero()
for r, c in zip(rows, cols):
    feature_name = vectorizer.get_feature_names_out()[c]

    # Library calculation
    # rank_bm25 stores precomputed doc lengths and frequencies
    # We simulate the scoring of a single term against a single doc
    doc_len = len(tokenized_corpus[r])
    freq = tokenized_corpus[r].count(feature_name)

    # Manual BM25 Formula (using the library's shared IDF)
    idf = bm25_lib.idf[feature_name]
    numerator = freq * (1.5 + 1)
    denominator = freq + 1.5 * (1 - 0.75 + 0.75 * (doc_len / bm25_trans.avg_dl_))
    expected_score = idf * (numerator / denominator)

    actual_score = X_dense[r, c]

    if not np.isclose(expected_score, actual_score, rtol=1e-6):
        print(f"Mismatch at Doc {r}, Term '{feature_name}'")
        print(f"Expected: {expected_score}, Got: {actual_score}")
        exit(1)

print("✅ Full matrix logic verified against manual calculation.")

--- 1. Setup Data ---
--- 2. Run Our BM25Transformer ---
--- 3. Run rank_bm25 (Ground Truth) ---
--- 4. Verify Specific Terms ---
Comparison for term 'dog':
Library: [0.46320012 0.56198687 0.         0.56198687 0.        ]
Ours:    [0.46320012 0.56198687 0.         0.56198687 0.        ]
✅ Single term match successful.

--- 5. Verify Full Matrix Consistency ---
✅ Full matrix logic verified against manual calculation.


In [25]:
%%time

Y = BM25Transformer(k1=8, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.1 s
Wall time: 25.4 s


np.float64(7755.2)

In [24]:
%%time

Y = BM25Transformer(k1=16, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.8 s
Wall time: 26.3 s


np.float64(6466.066666666667)

In [26]:
%%time

Y = BM25Transformer(k1=32, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.5 s
Wall time: 24.8 s


np.float64(5767.466666666666)

In [28]:
%%time

Y = BM25Transformer(k1=64, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.2 s
Wall time: 24.8 s


np.float64(5397.933333333333)

In [29]:
%%time

Y = BM25Transformer(k1=128, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24 s
Wall time: 24.6 s


np.float64(5215.066666666667)

In [34]:
%%time

Y = BM25Transformer(k1=256, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.5 s
Wall time: 24.8 s


np.float64(5124.366666666667)

In [35]:
%%time

Y = BM25Transformer(k1=512, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.9 s
Wall time: 25.5 s


np.float64(5085.766666666666)

In [36]:
%%time

Y = BM25Transformer(k1=1024, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.5 s
Wall time: 24.9 s


np.float64(5069.866666666667)

In [37]:
%%time

Y = BM25Transformer(k1=2048, b=0).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.2 s
Wall time: 25.8 s


np.float64(5060.933333333333)

In [44]:
%%time

Y = BM25Transformer(k1=2048, b=1).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 29.1 s
Wall time: 29.4 s


np.float64(5053.333333333333)

In [45]:
%%time

Y = BM25Transformer(k1=2048, b=1/2).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.2 s
Wall time: 25.4 s


np.float64(5053.433333333333)

In [41]:
%%time

Y = BM25Transformer(k1=2048, b=1/4).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 23.6 s
Wall time: 24 s


np.float64(5053.5)

In [42]:
%%time

Y = BM25Transformer(k1=2048, b=1/8).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.2 s
Wall time: 25.5 s


np.float64(5053.666666666667)

In [46]:
%%time

Y = TfidfTransformer(norm="l1").fit_transform(X2)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 8.3 s
Wall time: 12 s


np.float64(5031.133333333333)

In [47]:
%%time

Y = TfidfTransformer(use_idf=False).fit_transform(X2)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 5.84 s
Wall time: 6.03 s


np.float64(5008.6)

In [48]:
%%time

Y = X2

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 7.47 s
Wall time: 12.8 s


np.float64(5008.6)

In [49]:
%%time

Y = TfidfTransformer(smooth_idf=False).fit_transform(X2)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 7.31 s
Wall time: 7.4 s


np.float64(5031.133333333333)

In [40]:
%%time

Y = TfidfTransformer().fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 8.58 s
Wall time: 8.68 s


np.float64(5031.133333333333)

In [39]:
%%time

Y = TfidfTransformer(sublinear_tf=True).fit_transform(X)

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 10.4 s
Wall time: 10.7 s


np.float64(5718.4)

In [31]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.tocoo().nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [32]:
# alpha = .1 * .85
alpha = .1

Y = sparse_laplace_sppmi(X, alpha=alpha)

In [33]:
metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

np.float64(22603.266666666666)

In [42]:
%%time

bm25_trans = BM25Transformer(k1=1.2, b=0.75)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

30.0
33.1
32.8
CPU times: total: 23.9 s
Wall time: 24.5 s


np.float64(114.0)

In [41]:
%%time

bm25_trans = BM25Transformer(k1=2, b=0.75)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

32.7
33.1
32.9
CPU times: total: 24.8 s
Wall time: 25.3 s


np.float64(52.5)

In [45]:
%%time

bm25_trans = BM25Transformer(k1=2, b=1)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

32.7
33.1
32.9
CPU times: total: 24.6 s
Wall time: 24.8 s


np.float64(84.0)

In [44]:
%%time

bm25_trans = BM25Transformer(k1=2, b=.5)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

32.3
33.1
32.8
CPU times: total: 24.1 s
Wall time: 24.5 s


np.float64(35.5)

In [43]:
%%time

bm25_trans = BM25Transformer(k1=2, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

29.4
32.9
32.6
CPU times: total: 24.6 s
Wall time: 25.1 s


np.float64(18.5)

In [46]:
%%time

bm25_trans = BM25Transformer(k1=4, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

33.4
33.1
32.8
CPU times: total: 24.6 s
Wall time: 25.1 s


np.float64(10.5)

In [47]:
%%time

bm25_trans = BM25Transformer(k1=8, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

37.2
33.2
32.9
CPU times: total: 24.5 s
Wall time: 24.8 s


np.float64(5.5)

In [53]:
%%time

bm25_trans = BM25Transformer(k1=12, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

38.9
33.2
32.9
CPU times: total: 23.8 s
Wall time: 24.2 s


np.float64(4.5)

In [54]:
%%time

bm25_trans = BM25Transformer(k1=14, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

39.5
33.2
32.9
CPU times: total: 23.5 s
Wall time: 23.9 s


np.float64(4.0)

In [55]:
%%time

bm25_trans = BM25Transformer(k1=15, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

39.7
33.2
32.9
CPU times: total: 24.4 s
Wall time: 24.8 s


np.float64(3.5)

In [48]:
%%time

bm25_trans = BM25Transformer(k1=16, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

39.9
33.2
32.9
CPU times: total: 25 s
Wall time: 25.2 s


np.float64(3.5)

In [56]:
%%time

bm25_trans = BM25Transformer(k1=17, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.0
33.2
32.9
CPU times: total: 24.3 s
Wall time: 24.5 s


np.float64(3.5)

In [73]:
%%time

Y = BM25Transformer(k1=8, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 26.4 s
Wall time: 27.8 s


np.float64(6304.619565217391)

In [77]:
%%time

Y = BM25Transformer(k1=12, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.5 s
Wall time: 27.2 s


np.float64(6321.103773584906)

In [71]:
%%time

Y = BM25Transformer(k1=16, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.7 s
Wall time: 26.4 s


np.float64(5970.03125)

In [82]:
%%time

Y = BM25Transformer(k1=16, b=1/16).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 26.4 s
Wall time: 26.9 s


np.float64(6200.46685082873)

In [81]:
%%time

Y = BM25Transformer(k1=16, b=1/8).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.1 s
Wall time: 25.4 s


np.float64(6215.996987951808)

In [80]:
%%time

Y = BM25Transformer(k1=16, b=1/4).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.4 s
Wall time: 25.8 s


np.float64(6235.109271523179)

In [79]:
%%time

Y = BM25Transformer(k1=16, b=1/2).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.9 s
Wall time: 25.3 s


np.float64(6258.29044117647)

In [78]:
%%time

Y = BM25Transformer(k1=16, b=1).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 24.7 s
Wall time: 25.1 s


np.float64(6286.384297520661)

In [76]:
%%time

Y = BM25Transformer(k1=18, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.5 s
Wall time: 26.6 s


np.float64(6288.241758241758)

In [75]:
%%time

Y = BM25Transformer(k1=20, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.5 s
Wall time: 26.7 s


np.float64(6282.190789473684)

In [74]:
%%time

Y = BM25Transformer(k1=24, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 25.6 s
Wall time: 26.5 s


np.float64(6283.188524590164)

In [72]:
%%time

Y = BM25Transformer(k1=32, b=0).fit_transform(X)

for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

CPU times: total: 27.3 s
Wall time: 28.5 s


np.float64(6055.209677419355)

In [64]:
%%time

bm25_trans = BM25Transformer(k1=17, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.0
33.2
32.9
CPU times: total: 25.5 s
Wall time: 26.1 s


np.float64(3.5)

In [65]:
for idx in np.argsort(-Y[a].toarray())[:20]:
    print(idx2cat[idx])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)
Isbells (spotify:artist:14dULnNGmLKnS59BzNrHi4)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Matthew And The Atlas (spotify:artist:0lSENl3bteP8p2NbiSP7RM)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)
Axel Flovent (spotify:artist:6jn7W8NuX94FWZyeGlyCaJ)
S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Luke Sital-Singh (spotify:artist:3Lw97gGh8bp1MftsYmwJHG)
The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)
Night Beds (spotify:artist:533wKOfkJylNSi6ntO1wXd)
Tall Heights (spotify:artist:1OVaGC0CDZaxjcPxclSNmp)
Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)
Ivan & Alyosha (spotify:artist:3D1IyJznpDnWnnFrzjuWnh)
Matt C

In [66]:
for idx in np.argsort(-Y[b].toarray())[:20]:
    print(idx2cat[idx])

Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)
Odessa (spotify:artist:7xtlNrmdLZS2sqIkgWewi1)
RY X (spotify:artist:2KjAo6wVc9d2WcxdxSArpV)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Joseph (spotify:artist:5Wfvw7rDz7HA6gE2z6QhqO)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Broods (spotify:artist:5r5Va4lVQ1zjEfbJSrmCsS)
MUNA (spotify:artist:6xdRb2GypJ7DqnWAI2mHGn)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)
Bear's Den (spotify:artist:0nJaMZM8paoA5HEUTUXPqi)
Wet (spotify:artist:2i9uaNzfUtuApAjEf1omV8)
First Aid Kit (spotify:artist:21egYD1eInY6bGFcniCRT1)
Daughter (spotify:artist:46CitWgnWrvF9t70C2p1Me)
Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)


In [67]:
for idx in np.argsort(-Y[c].toarray())[:20]:
    print(idx2cat[idx])

Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
No. 4 (spotify:artist:24YjyPpqFQi1Oh7PQSBT3J)
Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)
Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
A-Laget (spotify:artist:2eJz0fbC9ZqBL3mGWPBvI5)
Cezinando (spotify:artist:504cl42JQLRqlZddfZ3S4z)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
Robyn (spotify:artist:6UE7nl9mha6s8z0wFQFIZ2)
Harry Styles (spotify:artist:6KImCVD70vtIoJWnq6nGn3)
Conan Gray (spotify:artist:4Uc8Dsxct0oMqx0P6i60ea)
Bodø Domkor (spotify:artist:6QQCeD7ZErV1xQnlyjaFHF)
Surferosa (spotify:artist:5WUYimkWakv41ZVVIq3BDr)
Astrid S (spotify:artist:3AVfmawzu83sp94QW7CEGm)
Herreløse (spotify:artist:3xvgxCYu3z6OR0MN1g1Nlu)
TIX (spotify:artist:6CawoDDP1IZUSGl4wSJGC9)
The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKp

In [68]:
for idx in np.argsort(-Y[d].toarray())[:20]:
    print(idx2cat[idx])

Mountain Man (spotify:artist:5kmPNusdo1mCTyz4u1uEGm)
Blessed Feathers (spotify:artist:0m2Xvn3JcV58sPCDV2GfSD)
Wiretree (spotify:artist:1Poa9x2nD1TLCxOalo71IE)
Bryan John Appleby (spotify:artist:3qPpwz6S0CbgMz9cGtn02l)
Fionn Regan (spotify:artist:0WJc0VDtzsLIk33XRB20Dy)
Harper Simon (spotify:artist:4fFL1ARvUJCoaZ3HK4SxdE)
Justin Vernon (spotify:artist:13rHmjtJmlIJ2aDyJc7CLV)
Fossil Collective (spotify:artist:0PhKXBGcwWwVcXCETFd92y)
Seabear (spotify:artist:6hLIT4e0yUtIa8DXwst4mi)
Chris and Thomas (spotify:artist:27VJZPUkuCYbgukhoDgFvj)
Rocky Votolato (spotify:artist:3OoiPKW9OUOaOS8l4DZXDq)
Bowerbirds (spotify:artist:4MoGkOmfK6oKi1Isc8Iv9m)
J. Tillman (spotify:artist:21XbnrbEMUTZelIfoV12hC)
Stornoway (spotify:artist:1UzKSuyOqKgUupcNNEtnF1)
Monsters Of Folk (spotify:artist:7wcYEfyBTrH0iT6J4PgSTj)
Dry the River (spotify:artist:5VIq5RHAbVUMF700vdwfYw)
Patrick Park (spotify:artist:4I1ZPDoEcZOul26MTXDhNO)
Rogue Valley (spotify:artist:1EbGAjTV50qpZ53jXTvmV7)
The Western States Motel (spotify:ar

In [69]:
for idx in np.argsort(-Y[e].toarray())[:20]:
    print(idx2cat[idx])

Bon Iver & St. Vincent (spotify:artist:5NexFJlYHMb5IC6z6ci3BA)
Justin Vernon (spotify:artist:13rHmjtJmlIJ2aDyJc7CLV)
Astronaut Husband (spotify:artist:31eyFwMQgaQ1ryWPeM7hbQ)
The Indians (spotify:artist:7MsNZABIIqhzypuBXQRN9H)
Scott Matthews (spotify:artist:6nrFqkvRs71iJWGnHJHa2O)
Joshua James (spotify:artist:7ewcIjvX2fkJPoPW2izgLF)
Unkle Bob (spotify:artist:3Bf4u6r96pGx1eIbaGqfvf)
Kyle Lionhart (spotify:artist:3VLXw7Phdo2mLlUoB5B59j)
Puzzle Muteson (spotify:artist:3PkGkJmTotXKubtTksWboK)
Nico Stai (spotify:artist:7pJUhzXnY3s96KGuQadfUQ)
DeYarmond Edison (spotify:artist:6arKuGbH3PuYfGN6yJ7RA9)
Isobel Anderson (spotify:artist:1tUN2f2byOej4LZdDq1UO9)
Ry Cuming (spotify:artist:51UlXSr2IrfDVa9xnJCuY6)
Jont (spotify:artist:27PAZpDiy0LBXUVkQ7D2UY)
The Black Atlantic (spotify:artist:33kge1mmCkHoYWJ4kJe6BC)
Garrett Kato (spotify:artist:4S3VOqqGguEZu3vbJMig4t)
Jack in Water (spotify:artist:2TR91e9wZCzK7QGBKtL8lb)
Fenne Lily (spotify:artist:7iPH2BRBF9wKa6ljxvdext)
Brett Bixby (spotify:artist:5tJ

In [60]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1/1000)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.1
33.2
32.9
CPU times: total: 30.7 s
Wall time: 31.5 s


np.float64(3.5)

In [61]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1/100)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.1
33.2
32.9
CPU times: total: 26.3 s
Wall time: 27.2 s


np.float64(3.5)

In [62]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1/10)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.8
33.2
32.9
CPU times: total: 25 s
Wall time: 25.5 s


np.float64(6.0)

In [63]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1/4)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

KeyboardInterrupt: 

In [ ]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1/2)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
%%time

bm25_trans = BM25Transformer(k1=17, b=3/4)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [ ]:
%%time

bm25_trans = BM25Transformer(k1=17, b=1)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

In [57]:
%%time

bm25_trans = BM25Transformer(k1=18, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.2
33.2
32.9
CPU times: total: 24.7 s
Wall time: 25 s


np.float64(4.0)

In [58]:
%%time

bm25_trans = BM25Transformer(k1=19, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

KeyboardInterrupt: 

In [52]:
%%time

bm25_trans = BM25Transformer(k1=20, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.4
33.2
32.9
CPU times: total: 23.7 s
Wall time: 24.1 s


np.float64(4.5)

In [51]:
%%time

bm25_trans = BM25Transformer(k1=24, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.7
33.2
32.9
CPU times: total: 23.8 s
Wall time: 24.3 s


np.float64(10.5)

In [49]:
%%time

bm25_trans = BM25Transformer(k1=32, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

40.9
33.2
32.9
CPU times: total: 25.3 s
Wall time: 25.6 s


np.float64(21.5)

In [50]:
%%time

bm25_trans = BM25Transformer(k1=64, b=0)
X_bm25 = bm25_trans.fit_transform(X)

Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

KeyboardInterrupt: 

In [37]:
%%time

bm25_trans = BM25Transformer(k1=1.5, b=0.75)
X_bm25 = bm25_trans.fit_transform(X)

CPU times: total: 25.4 s
Wall time: 25.6 s


In [38]:
Y = X_bm25

import pandas as pd

for corr_type in ["pearson", "spearman", "kendall"]:
    print(round(100*pd.DataFrame({
        "a": Y[b].toarray(),
        "b": Y[c].toarray(),
    }).corr(corr_type)["a"]["b"], 1))

metrics = [
#     get_metric(a, b, Y),
#     get_metric(b, a, Y),
    
    get_metric(b, c, Y),
    get_metric(c, b, Y),
]

np.mean(metrics)

31.2
33.1
32.8


np.float64(90.0)

In [40]:
for idx in np.argsort(-Y[a].toarray())[:20]:
    print(idx2cat[idx])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)
Isbells (spotify:artist:14dULnNGmLKnS59BzNrHi4)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)
Matthew And The Atlas (spotify:artist:0lSENl3bteP8p2NbiSP7RM)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)
Tall Heights (spotify:artist:1OVaGC0CDZaxjcPxclSNmp)
Matt Corby (spotify:artist:7CIW23FQUXPc1zebnO1TDG)
Night Beds (spotify:artist:533wKOfkJylNSi6ntO1wXd)
Axel Flovent (spotify:artist:6jn7W8NuX94FWZyeGlyCaJ)
Luke Sital-Singh (spotify:artist:3Lw97gGh8bp1MftsYmwJHG)
Roo Panes (spotify:artist:0XHM5ZNJDU8e4CfbWMeSzC)
Little May (spotify:artist:0TjAAwE04BeoSeOpJIakYH)
The Careful Ones (